# Table S3. Held-out R² for predicting the settings chosen by CEM

The CEM schedules of the 200 evaluation batches were replayed in pyIndPenSim, and the state just before each 12 h
decision was recorded (3,726 decisions; data/w77_cem_rows.npz). The setting CEM chose for each operating variable is
predicted from (A) the sixteen on-line channels used by PPO and (B) the same channels plus six off-line variables
- (glucose, PAA, biomass, penicillin, ammonia, viscosity), with gradient-boosted trees and five-fold cross-validation
grouped by batch. The saved result of the original run is data/w77_cem_imitate.json.

In [1]:
import sys; sys.path.insert(0, '..')          # repository root, for paths.py
from paths import DATA, OUTD                   # data/ and output/

In [2]:
import os, json
os.environ.setdefault('OMP_NUM_THREADS', '1')      # as in the original run; many small fits are slower multi-threaded
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

z = np.load(DATA + 'w77_cem_rows.npz')
B, XA, XO, Y, LEV, OFF = z['batch'], z['XA'], z['XO'], z['Y'], list(z['lev']), list(z['off'])
XB = np.hstack([XA, XO]); gkf = GroupKFold(5)
NAME = {'Fs': 'F_s', 'Fpaa': 'F_PAA', 'Foil': 'F_oil', 'Fg': 'F_g', 'Fw': 'F_w', 'Fdis': 'F_dis',
        'RPM': 'Agitation speed', 'pressure': 'Vessel pressure', 'pH_sp': 'pH set-point', 'T_sp': 'Temperature set-point'}


def cv(X, y):
    p = np.zeros_like(y)
    for tr, te in gkf.split(X, y, B):
        p[te] = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(X[tr], y[tr]).predict(X[te])
    return r2_score(y, p)


L = [f'Table S3. Held-out R2 for predicting the settings chosen by CEM ({len(Y):,} decisions, {len(set(B))} batches)', '',
     f'{"Operating variable":24s}{"On-line channels":>18s}{"On-line and off-line":>22s}']
res = {}
for j, nm in enumerate(LEV):
    if np.std(Y[:, j]) < 1e-9:
        continue
    ra, rb = cv(XA, Y[:, j]), cv(XB, Y[:, j]); res[nm] = (ra, rb)
    L.append(f"{NAME[nm]:24s}{ra:18.3f}{rb:22.3f}")
L += ['', 'Adding one off-line variable to the on-line channels (R2):']
for nm in ('Fs', 'Fpaa'):
    j = LEV.index(nm)
    L.append(f'  {NAME[nm]:6s} ' + ' . '.join(f'{o} {cv(np.hstack([XA, XO[:, [i]]]), Y[:, j]):.3f}' for i, o in enumerate(OFF)))
txt = '\n'.join(L); print(txt); open(OUTD + 'tableS3.txt', 'w').write(txt + '\n')

Table S3. Held-out R2 for predicting the settings chosen by CEM (3,726 decisions, 200 batches)

Operating variable        On-line channels  On-line and off-line
F_s                                  0.521                 0.512
F_PAA                                0.496                 0.508
F_oil                                0.442                 0.454
F_g                                  0.267                 0.295
F_w                                  0.367                 0.377
F_dis                                0.169                 0.174
Agitation speed                      0.327                 0.315
Vessel pressure                      0.075                 0.087
pH set-point                         0.264                 0.275
Temperature set-point                0.294                 0.319

Adding one off-line variable to the on-line channels (R2):
  F_s    S 0.506 . PAA 0.519 . X 0.513 . P 0.522 . NH3 0.512 . visc 0.517
  F_PAA  S 0.498 . PAA 0.499 . X 0.487 . P 0.490 . NH3 

1019